In [ ]:
# import tensorflow_decision_forests as tfdf
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
import os

import numpy as np
# import tensorflow as tf
# print(tf.test.is_gpu_available())
import pandas as pd
import os
from tqdm import tqdm
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import PIL
import pandas as pd
from sklearn.utils import shuffle
from sklearn.preprocessing import MinMaxScaler,StandardScaler,LabelEncoder,QuantileTransformer,PowerTransformer

feature_names=['f4','f7','f9','f10','f11','f13','f17','f22','f23','f24']


Mounted at /content/drive


In [ ]:


path='/content/drive/MyDrive/Image/data_normalization'
Boulevard=pd.read_csv(path+'/Boulevard_norm.csv')

Defense=pd.read_csv(path+'/Defense_norm.csv')

FullNantes=pd.read_csv(path+'/FullNantes_norm.csv')

Paris=pd.read_csv(path+'/Paris_norm.csv')

Toulouse=pd.read_csv(path+'/Toulouse_norm.csv')


Nantes=pd.read_csv(path+'/Nantes_norm.csv')

train_data=pd.concat([Nantes,Paris,Toulouse,FullNantes,Boulevard])
val_data=   Defense


In [ ]:

x_train=train_data[feature_names].values
y_train=train_data['LosNLos'].values.astype('float32')
x_val=val_data[feature_names].values
y_val=val_data['LosNLos'].values.astype('float32')
# OneHotEncoder Label
# from sklearn.preprocessing import OneHotEncoder
# enc = OneHotEncoder(handle_unknown='ignore')
# enc.fit(y_train.reshape(-1, 1))
# y_train=enc.transform(y_train.reshape(-1, 1)).toarray()
# y_val=enc.transform(y_val.reshape(-1, 1)).toarray()

In [ ]:
import tensorflow as tf

def inception(x,n_map):
   x = tf.keras.layers.Conv2D(n_map, (1, 1), strides=(1, 1), padding='same')(x)
   x = tf.keras.layers.LeakyReLU()(x)

   x1= tf.keras.layers.Conv2D(n_map, (3, 3), strides=(1, 1), padding='same')(x)
   x1 = tf.keras.layers.LeakyReLU()(x1)

   x2= tf.keras.layers.Conv2D(n_map, (5, 5), strides=(1, 1), padding='same')(x)
   x2 = tf.keras.layers.LeakyReLU()(x2)

   x3= tf.keras.layers.Conv2D(n_map, (7, 7), strides=(1, 1), padding='same')(x)
   x3 = tf.keras.layers.LeakyReLU()(x3)

   output=tf.keras.layers.Add()([x,x1,x2,x3])
   return  output
def Carnet(raw_features):

    x = tf.keras.layers.Reshape((1, 1,10))(raw_features)

    x = tf.keras.layers.Conv2DTranspose(10, (1, 1), strides=(3, 3), padding='same', use_bias=False)(x)
    x = tf.keras.layers.LeakyReLU()(x)

    x = tf.keras.layers.Conv2DTranspose(5, (3, 3), strides=(2, 2), padding='same', use_bias=False)(x)
    x = tf.keras.layers.LeakyReLU()(x)

    x = tf.keras.layers.Conv2DTranspose(1, (3, 3), strides=(2, 2), padding='same', use_bias=False, activation='tanh')(x)


    x = inception(x,8)
    x = tf.keras.layers.MaxPool2D((2, 2), strides=(2,2), padding='same')(x)

    x = inception(x,16)
    # x = tf.keras.layers.MaxPool2D((2, 2), strides=(2,2), padding='same')(x)
    # x = inception(x,32)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(32)(x)
    x = tf.keras.layers.LeakyReLU()(x)

    x = tf.keras.layers.Dropout(0.4)(x)
    x = tf.keras.layers.Dense(1, activation='sigmoid')(x)
    model = tf.keras.models.Model(inputs=raw_features, outputs=x, name='CarNet')


    return model


In [ ]:
raw_features = tf.keras.layers.Input(shape=(10))
model=Carnet(raw_features)
# model.load_weights('/content/drive/MyDrive/Image/ParisImg_nettoyé.hdf5')
model.summary()

Model: "CarNet"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_3 (InputLayer)        [(None, 10)]                 0         []                            
                                                                                                  
 reshape_2 (Reshape)         (None, 1, 1, 10)             0         ['input_3[0][0]']             
                                                                                                  
 conv2d_transpose_6 (Conv2D  (None, 3, 3, 10)             100       ['reshape_2[0][0]']           
 Transpose)                                                                                       
                                                                                                  
 leaky_re_lu_22 (LeakyReLU)  (None, 3, 3, 10)             0         ['conv2d_transpose_6[0][0

In [ ]:
model_save_path='/content/drive/MyDrive/Image/'


# ensemble_nn_only.summary()
model_checkpoint = tf.keras.callbacks.ModelCheckpoint(model_save_path  + 'defense_nettoyé103.hdf5',
                                                          monitor='val_accuracy', verbose=1, save_best_only=True)

In [ ]:
opti_fun = tf.keras.optimizers.SGD(learning_rate =1e-2)
model.compile(optimizer=opti_fun, loss='binary_crossentropy', metrics=['accuracy'])
history = model.fit(x=x_train,
                    y=y_train,
                                  epochs=50, validation_data=(x_val,y_val),
                                  # class_weight=d_class_weights,

                                  verbose = 1,
                                  batch_size=256,
                                  callbacks=[model_checkpoint],shuffle=True)

Epoch 1/50
5014/5016 [============================>.] - ETA: 0s - loss: 0.4931 - accuracy: 0.7753
Epoch 1: val_accuracy improved from -inf to 0.80258, saving model to /content/drive/MyDrive/Image/defense_nettoyé103.hdf5


/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


5016/5016 [==============================] - 41s 8ms/step - loss: 0.4931 - accuracy: 0.7753 - val_loss: 0.4229 - val_accuracy: 0.8026
Epoch 2/50
5015/5016 [============================>.] - ETA: 0s - loss: 0.4248 - accuracy: 0.8202
Epoch 2: val_accuracy improved from 0.80258 to 0.80704, saving model to /content/drive/MyDrive/Image/defense_nettoyé103.hdf5
5016/5016 [==============================] - 39s 8ms/step - loss: 0.4248 - accuracy: 0.8202 - val_loss: 0.4205 - val_accuracy: 0.8070
Epoch 3/50
5015/5016 [============================>.] - ETA: 0s - loss: 0.4135 - accuracy: 0.8240
Epoch 3: val_accuracy did not improve from 0.80704
5016/5016 [==============================] - 38s 8ms/step - loss: 0.4135 - accuracy: 0.8240 - val_loss: 0.4333 - val_accuracy: 0.8039
Epoch 4/50
5013/5016 [============================>.] - ETA: 0s - loss: 0.4058 - accuracy: 0.8254
Epoch 4: val_accuracy did not improve from 0.80704
5016/5016 [==============================] - 38s 8ms/step - loss: 0.4058 - ac

KeyboardInterrupt: 

In [ ]:
opti_fun = tf.keras.optimizers.SGD(learning_rate =1e-3)
model.compile(optimizer=opti_fun, loss='binary_crossentropy', metrics=['accuracy'])
history = model.fit(x=x_train,
                    y=y_train,
                                  epochs=500, validation_data=(x_val,y_val),
                                  # class_weight=d_class_weights,

                                  verbose = 1,
                                  batch_size=128,
                                  callbacks=[model_checkpoint],shuffle=True)

Epoch 1/500
10031/10031 [==============================] - ETA: 0s - loss: 0.4232 - accuracy: 0.8210
Epoch 1: val_accuracy did not improve from 0.81320
10031/10031 [==============================] - 78s 7ms/step - loss: 0.4232 - accuracy: 0.8210 - val_loss: 0.4224 - val_accuracy: 0.8108
Epoch 2/500
10029/10031 [============================>.] - ETA: 0s - loss: 0.4203 - accuracy: 0.8231
Epoch 2: val_accuracy did not improve from 0.81320
10031/10031 [==============================] - 73s 7ms/step - loss: 0.4203 - accuracy: 0.8231 - val_loss: 0.4329 - val_accuracy: 0.8072
Epoch 3/500
10031/10031 [==============================] - ETA: 0s - loss: 0.4180 - accuracy: 0.8244
Epoch 3: val_accuracy did not improve from 0.81320
10031/10031 [==============================] - 73s 7ms/step - loss: 0.4180 - accuracy: 0.8244 - val_loss: 0.4289 - val_accuracy: 0.8089
Epoch 4/500
10031/10031 [==============================] - ETA: 0s - loss: 0.4161 - accuracy: 0.8252
Epoch 4: val_accuracy did not impro

KeyboardInterrupt: 

In [ ]:
from sklearn.metrics import confusion_matrix,accuracy_score,classification_report
model.load_weights('/content/drive/MyDrive/Image/defense_nettoyé10.hdf5')
val_predict=np.where(model.predict(x_val)>0.5,1,0)


4528/4528 [==============================] - 16s 4ms/step


In [ ]:

print('The accuracy of val is:',accuracy_score(y_val,val_predict))
confusion_matrix_result = confusion_matrix(val_predict,y_val)
print('The confusion matrix result:\n',confusion_matrix_result)

print(
    f"Classification report for classifier {model.name}:\n"
    f"{classification_report(y_val, val_predict)}\n"
)

The accuracy of val is: 0.8247056225065915
The confusion matrix result:
 [[107842  15693]
 [  9704  11643]]
Classification report for classifier CarNet:
              precision    recall  f1-score   support

         0.0       0.87      0.92      0.89    117546
         1.0       0.55      0.43      0.48     27336

    accuracy                           0.82    144882
   macro avg       0.71      0.67      0.69    144882
weighted avg       0.81      0.82      0.82    144882


